# Classical ML: dummy, logistic regression, BM25, MLP, Naive Bayes, one-class SVM (notebook 02)

Part A walks through one seed-0 split with the pipeline from the course slides: dummy
baseline, TF-IDF logistic regression with a threshold picked on validation, BM25, an MLP,
Complement Naive Bayes and an unsupervised one-class SVM, then explanations and
error reading. Only training and validation rows
are used there. Part B runs the tuned experiment for all five split seeds (finished runs are
reused from `outputs/classical/`, so this is fast unless you delete them); the test split is
scored once per (model, seed) inside the run. Comparing and plotting the saved results is
notebook 04.

Run time: about 5 minutes for part A; part B is under a minute when the saved runs exist, or
about an hour if it has to recompute all 30 runs.

In [ ]:
import sys, os; os.chdir(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else None; sys.path.insert(0, "."); import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from modules import data_helper as dh
from modules import features as ft
from modules import models as md
from modules import eval_helper as ev
from modules import explain_helper as eh
from sklearn.inspection import permutation_importance
from modules import classical_runs as cr
from sklearn.metrics import ConfusionMatrixDisplay, PrecisionRecallDisplay, precision_recall_curve

DATA = "../data/fake_job_postings.csv"

In [ ]:
SEED = 0
SEEDS = [0, 1, 2, 3, 4]
SPLIT_PATH = "splits/seed0.npz"
OUT_CLASSICAL = "outputs/classical"

dh.seed_everything(SEED)
sns.set_theme(style="whitegrid")

# Part A: one split, validation only

## Setup: the seed-0 split

Load, clean, add features, and reuse the saved seed-0 split (Session 5: split
first, then fit on the training rows only).

In [ ]:
# Load, clean (drop duplicates), add features.
df = ft.add_engineered_features(dh.load_processed())

# Reuse the saved split, or create and save it the first time.
if os.path.exists(SPLIT_PATH):
    splits = dh.load_splits(SPLIT_PATH, df)
else:
    splits = dh.make_splits(df, seed=SEED)
    dh.save_splits(SPLIT_PATH, splits, df)
assert not set(splits["train"]) & set(splits["val"]), "train/val row overlap"

# Frames and labels for the two splits this notebook uses (test stays untouched).
train = df.iloc[splits["train"]]
val = df.iloc[splits["val"]]
y_train = train["fraudulent"].to_numpy()
y_val = val["fraudulent"].to_numpy()

In [ ]:
y_all = df["fraudulent"].to_numpy()
for name in ("train", "val", "test"):
    idx = splits[name]
    print(f"{name:5s}: {len(idx):6d} rows, {int(y_all[idx].sum()):4d} fraud ({y_all[idx].mean():.1%})")
print("test rows are listed for completeness; this notebook never scores them")

## One helper for every model

The same three steps each time: fit on train, pick the decision threshold on
validation (never on test), then report train and validation side by side so the
generalisation gap is visible (Session 6). `ev.compare_table` names its second split
"test"; we pass validation metrics there and rename the columns below, so the test
rows are never touched.

In [ ]:
# Everything the later cells need, keyed by model name.
fitted, thresholds, scores, results = {}, {}, {}, {}


def report(name, model):
    """Score a fitted model; the threshold is picked on validation only."""
    # Validation scores, then the threshold chosen on them.
    s_val = ev.score_of(model, val)
    # A dummy scores every ad the same, so it keeps its own predictions (threshold 0.5).
    thr = 0.5 if name.startswith("dummy") else ev.pick_threshold(y_val, s_val)

    # Train and validation metrics at that threshold, stored for the comparison table.
    results[name] = {"train": ev.evaluate(y_train, ev.score_of(model, train), thr),
                     "test": ev.evaluate(y_val, s_val, thr)}
    fitted[name], thresholds[name], scores[name] = model, thr, s_val

    # One small table for display.
    out = pd.DataFrame(results[name]).T.rename(index={"test": "val"})  # "test" slot holds validation
    out = out[["pr_auc", "roc_auc", "f1", "precision", "recall"]]
    out.insert(0, "model", name)
    return out

## The dummy baseline (Sessions 6/7)

Every model has to beat the dumb reference first: `dummy_majority` always predicts the
majority class (real). It reaches about 95% accuracy while catching no fraud at all, which is why
accuracy is not our metric; its PR-AUC equals the fraud rate (about 0.049), the floor for PR-AUC.
Its score is the same for every ad, so a tuned threshold would mean nothing; it keeps 0.5.

In [ ]:
frames = [report(name, est.fit(train, y_train))
          for name, est in md.get_baselines(SEED).items()]
pd.concat(frames)

## Logistic regression on TF-IDF + engineered features

`modules/models.py` builds one Pipeline: TF-IDF over the joined text, one-hot and
ordinal encoded metadata, scaled numeric flags, then a `class_weight="balanced"`
logistic regression.

In [ ]:
report("logreg", md.get_models(SEED)["logreg"].fit(train, y_train))

### Confusion matrix on validation

In [ ]:
# Hard predictions at the validation-chosen threshold, drawn with sklearn's display.
pred_val = (scores["logreg"] >= thresholds["logreg"]).astype(int)
ConfusionMatrixDisplay.from_predictions(y_val, pred_val, display_labels=["real", "fraud"],
                                        colorbar=False)
plt.title("Validation confusion matrix (logreg)")
plt.show()

Reading: rows are the actual class, columns the predicted one; the diagonal holds
the correct calls.

### The threshold trade-off

Session 6: with an imbalanced target there is no free lunch between precision and
recall; the threshold is a business decision, and we pick it on validation.

In [ ]:
# Precision and recall at every distinct score (the last point has no threshold).
prec, rec, thr_grid = precision_recall_curve(y_val, scores["logreg"])

# Two curves plus the threshold that was actually chosen.
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(thr_grid, prec[:-1], label="precision")
ax.plot(thr_grid, rec[:-1], label="recall")
ax.axvline(thresholds["logreg"], color="grey", linestyle="--", label="chosen threshold")
ax.set_xlabel("threshold on the fraud score")
ax.set_ylabel("metric value")
ax.set_title("Precision and recall vs threshold (logreg, validation)")
ax.legend()
plt.show()

Reading: the dashed line is the threshold picked on validation; moving it right can
only shrink the set of flagged ads.

## BM25 instead of TF-IDF

BM25 is the ranking function behind classic search engines. It differs from TF-IDF
in two ways: repeating a word has diminishing returns (saturating term frequency),
and long ads are discounted relative to the average ad length (length normalisation).
The vocabulary and everything else in the pipeline are identical, so any difference
in the numbers comes from the weighting alone.

In [ ]:
report("logreg_bm25", md.get_models(SEED)["logreg_bm25"].fit(train, y_train))

## MLP on the same features

A small sklearn neural network. It has no `class_weight`, so we rely on the ranking
of its scores plus the threshold picked on validation (Session 5).

In [ ]:
report("mlp", md.get_models(SEED)["mlp"].fit(train, y_train))

## Complement Naive Bayes

The classic text-classification baseline: word counts per class, with no feature interactions.
Complement NB is the variant designed for imbalanced classes (it learns each class from the
documents of all *other* classes). It needs non-negative inputs, which TF-IDF, one-hot flags and
the scaled counts are.

In [ ]:
report("cnb", md.get_models(SEED)["cnb"].fit(train, y_train))

## One-class SVM: can fraud be found without labels?

Anomaly detection view: learn what a *real* ad looks like from the real training ads only, then
flag ads that do not fit. The detector never sees a fraud example; the labels are used only to
leave the frauds out of its training data and, in part B, to choose its settings.

Two choices make this a fair test. First, the input is the **text only**: in the full feature
matrix four numeric columns (lengths, capitals) make up about half of each row's length, and an
RBF kernel works on distances, so it would mostly compare ad lengths. Second, the RBF kernel needs
short dense input, so the TF-IDF text is reduced to 100 dimensions with **PCA** (the `arpack`
solver works on the sparse matrix directly) and each row is rescaled to unit length. The score is
how far an ad lies outside the region of the real ads.

In [ ]:
report("ocsvm", md.get_models(SEED)["ocsvm"].fit(train, y_train))

### gamma decides how "unusual" is measured

`gamma` sets how local the RBF kernel is. Grouped 3-fold CV on the training rows only, for a few
values (this takes a few minutes):

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold, cross_val_score

folds = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=SEED)
for gamma in [0.3, 3, 30, 300]:
    pipe = md.get_models(SEED)["ocsvm"].set_params(clf__detector__svm__gamma=gamma)
    pr = cross_val_score(pipe, train, y_train, groups=train["group"], cv=folds,
                         scoring="average_precision")
    roc = cross_val_score(pipe, train, y_train, groups=train["group"], cv=folds, scoring="roc_auc")
    print(f"gamma {gamma:>5}: CV PR-AUC {pr.mean():.3f}  ROC-AUC {roc.mean():.3f}")

Reading: with a broad kernel (small gamma) the ROC-AUC is far **below** 0.5, so frauds look *more*
typical than real ads: many are repeated templates that sit in dense regions. With a local kernel
(gamma around 30) the detector asks whether an ad has a close neighbour among the real ads, and
frauds often do not. The fraud base rate (PR-AUC floor) is about 0.049.

Is the remaining gap to the supervised models a lack of information in these features, or a limit
of the anomaly view? Check: the same text, the same PCA, but a supervised logistic regression
that is allowed to see the labels.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

pca_lr = Pipeline([("pre", ColumnTransformer([("text", ft.make_tfidf(), "text")])),
                   ("pca", PCA(n_components=100, svd_solver="arpack", random_state=SEED)),
                   ("clf", LogisticRegression(class_weight="balanced", max_iter=2000))])
report("pca_logreg (check)", pca_lr.fit(train, y_train))

Reading: if the supervised model on the very same features scores far above the one-class SVM,
the information is there and the labels are what the anomaly view is missing.

## All models side by side

Train versus validation PR-AUC per model; the gap column is the generalisation gap.
This is validation, not test: part B scores the test split once per (model, seed).

In [ ]:
table = ev.compare_table(results).rename(columns=lambda c: c.replace("test_", "val_"))
table.sort_values("val_pr_auc", ascending=False)

## Hyper-parameter tuning (switched off)

Part B below runs the full random search over all models and five seeds:
grouped cross-validation on the training rows only, scoring PR-AUC (Session 7). The
flag below runs a tiny live version (`n_iter=3, cv=3`); keep it False during the
demo.

In [ ]:
RUN_TUNING = False  # flip to True to run a small random search live (slow)

if RUN_TUNING:
    # Grouped CV on the training rows so duplicate descriptions cannot leak across folds.
    search = md.tune(md.get_models(SEED)["logreg"], md.PARAM_GRIDS["logreg"],
                     train, y_train, train["group"].to_numpy(),
                     n_iter=3, cv=3, seed=SEED)
    print("best params:", search.best_params_)
    print("CV PR-AUC:", round(float(search.best_score_), 3))

## Which words drive the logreg?

Session 6: read the model. `top_linear_terms` maps the coefficients back to feature
names recovered from the fitted ColumnTransformer.

In [ ]:
# The 15 strongest terms on each side, sorted so the bars read from blue to red.
terms = eh.top_linear_terms(fitted["logreg"], k=15).sort_values("coef")

# Horizontal bars: red pushes towards fraud, blue towards real.
fig, ax = plt.subplots(figsize=(8, 7))
ax.barh(terms["term"], terms["coef"],
        color=np.where(terms["coef"] > 0, "tab:red", "tab:blue"))
ax.set_xlabel("logistic regression coefficient")
ax.set_title("Strongest fraud (red) and real-job (blue) terms")
plt.show()

Reading: red bars push an ad towards fraud and blue bars towards real; bar length is
the coefficient, not the importance of a whole input column.

## Which columns matter? Permutation importance

Shuffle one raw input column at a time and see how much validation PR-AUC drops
(Session 6). We use a 1000-row validation sample and a single repeat to keep it
quick.

In [ ]:
# Small validation sample and one repeat keep this quick. sklearn shuffles each raw column
# of the frame in turn and measures the drop in PR-AUC of the whole fitted pipeline.
val_sample = val.sample(min(1000, len(val)), random_state=SEED)
perm = permutation_importance(fitted["logreg"], val_sample, val_sample["fraudulent"].to_numpy(),
                              scoring="average_precision", n_repeats=1, random_state=SEED)

# Top 10 columns as a bar chart.
top = pd.Series(perm.importances_mean, index=val_sample.columns).nlargest(10).iloc[::-1]
fig, ax = plt.subplots(figsize=(7, 5))
ax.barh(top.index, top.values)
ax.set_xlabel("drop in validation PR-AUC when the column is shuffled")
ax.set_title("Permutation importance (validation sample, 1 repeat)")
plt.show()

Reading: the longer the bar, the more the model leaned on that single input column;
near-zero bars mean the column was barely used.

## Read the errors (Session 6)

The five most confident false positives (flagged, but real) and false negatives
(not flagged, but fraud) of the logreg on validation.

In [ ]:
errors = eh.show_errors(val, y_val, scores["logreg"], thresholds["logreg"], k=5)
for _, e in errors.iterrows():
    print(f"[{e['error_type']}] score={e['score']:.3f}  title: {e['title']}")
    print(e["description"], "\n")

## Per-group check: employment type

Session 6: a model can look fine on average and still fail one group. Recall and
precision by employment type at the logreg's validation threshold.

In [ ]:
# One row per prediction, then counts per employment type (missing is its own group).
pred_val = (scores["logreg"] >= thresholds["logreg"]).astype(int)
g = pd.DataFrame({"group": val["employment_type"].fillna("missing").to_numpy(),
                  "y": y_val, "pred": pred_val})
g["tp"] = g["y"] & g["pred"]
tab = g.groupby("group").agg(count=("y", "size"), positives=("y", "sum"),
                             flagged=("pred", "sum"), tp=("tp", "sum"))

# Recall = caught frauds / all frauds; precision = caught frauds / flagged ads.
tab["recall"] = tab["tp"] / tab["positives"]
tab["precision"] = tab["tp"] / tab["flagged"]
tab[["count", "positives", "recall", "precision"]]

Reading: rows with few ads have noisy recall and precision; the count column says
how much to trust each row.

# Part B: tuned runs over five split seeds

For each seed and model `cr.run_classical` tunes on the training rows (random search, 4
candidates, 3-fold grouped CV, scoring PR-AUC), refits, picks the threshold on validation, and
scores validation and test once. One JSON + NPZ pair per (model, seed) goes to
`outputs/classical/`; pairs that already exist are skipped. Only validation numbers are printed.

In [ ]:
RUN_CLASSICAL = True  # False: only read the saved results below
if RUN_CLASSICAL:
    df_all = ft.add_engineered_features(dh.load_processed())
    cr.run_classical(df_all, SEEDS, out_dir=OUT_CLASSICAL, n_iter=4, cv=3)